# Run tornado + whirlpool and evaluate marginal abatement cost

This notebook is the **single entry point** for the pipeline. Set all parameters in **Section 1**, then run the cells in order.

1. **Run CBA** (`scr/cba_uganda_tornado_whirlpool.py`) → writes `cba_results_ssp_modeling_tornado_<country>.csv` and `cba_results_ssp_modeling_whirlpool_<country>.csv` to `data/input/tornado` and `data/input/whirlpool`.
2. **Run tornado process** (`scr/process_tornado.py`) → writes `data/output/tornado/tornado_plot.csv`.
3. **Run whirlpool process** (`scr/process_whirlpool.py`) → writes `data/output/whirlpool/tornado_plot_whirlpool.csv`.
4. **Evaluation** → builds `marginal_abatement_cost_evaluation.csv` and summary from the `marginal_total_abatement_cost_(USD/tCO2e)` column.

**Reusable for other countries/repos:** Change `COUNTRY`, `PATH_RUN_OUTPUT`, and (if needed) `TORNADO_PLOT_DIR` or paths below; the same scripts run with the new parameters. Run this notebook from **project root**.

**To only refresh evaluation** (after running the three scripts): run Section 1 (Parameters) and Section 4 (Evaluation).

## 1. Parameters

Edit the values below for your country and run. All three scripts (CBA, tornado, whirlpool) are called with these parameters.

In [12]:
import sys
import subprocess
import pathlib
import pandas as pd
import numpy as np

# ---------- Paths (change for other repos / runs) ----------
PROJECT_ROOT = pathlib.Path.cwd()
if not (PROJECT_ROOT / "ssp_modeling").is_dir():
    for _ in range(5):
        PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
        if (PROJECT_ROOT / "ssp_modeling").is_dir():
            break

TORNADO_PLOT_DIR = PROJECT_ROOT / "ssp_modeling" / "cb" / "tornado_plot"
PATH_RUN_OUTPUT = PROJECT_ROOT / "ssp_modeling/ssp_run_output/sisepuede_summary_results_run_sisepuede_run_2026-02-18T21;36;42.734194"
PATH_CB_INPUT = TORNADO_PLOT_DIR / "data" / "input"
PATH_OUTPUT = TORNADO_PLOT_DIR / "data" / "output"

# ---------- Country and run (change for other countries) ----------
COUNTRY = "uganda"
EMISSIONS_YEAR = 2019

# ---------- Strategy selection ----------
# None = run ALL strategies. A string = run only that strategy_code (BASE is always included).
RUN_STRATEGY_TORNADO = "AGRC:DEC_CH4_RICE"
RUN_STRATEGY_WHIRLPOOL = "WHIRLPOOL:TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ"

# ---------- Script paths and output paths (derived) ----------
CBA_SCRIPT = TORNADO_PLOT_DIR / "scr" / "cba_uganda_tornado_whirlpool.py"
PROCESS_TORNADO_SCRIPT = TORNADO_PLOT_DIR / "scr" / "process_tornado.py"
PROCESS_WHIRLPOOL_SCRIPT = TORNADO_PLOT_DIR / "scr" / "process_whirlpool.py"
INPUT_TORNADO = PATH_CB_INPUT / "tornado"
INPUT_WHIRLPOOL = PATH_CB_INPUT / "whirlpool"
OUTPUT_TORNADO_CSV = PATH_OUTPUT / "tornado" / "tornado_plot.csv"
OUTPUT_WHIRLPOOL_CSV = PATH_OUTPUT / "whirlpool" / "tornado_plot_whirlpool.csv"
MAC_COL = "marginal_total_abatement_cost_(USD/tCO2e)"
EVALUATION_OUTPUT_DIR = PATH_OUTPUT
EVALUATION_CSV = EVALUATION_OUTPUT_DIR / "marginal_abatement_cost_evaluation.csv"
EVALUATION_SUMMARY_CSV = EVALUATION_OUTPUT_DIR / "marginal_abatement_cost_summary.csv"

## 2. Run CBA for tornado and whirlpool

In [13]:
cmd = [sys.executable, str(CBA_SCRIPT), "both", "--country", COUNTRY, "--path-run-output", str(PATH_RUN_OUTPUT), "--path-cb-output", str(PATH_CB_INPUT)]
if RUN_STRATEGY_TORNADO is not None:
    cmd.extend(["--tornado-strategy", RUN_STRATEGY_TORNADO])
if RUN_STRATEGY_WHIRLPOOL is not None:
    cmd.extend(["--whirlpool-strategy", RUN_STRATEGY_WHIRLPOOL])

result = subprocess.run(cmd, cwd=str(PROJECT_ROOT), capture_output=True, text=True)
print(result.stdout or "")
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"CBA script exited with code {result.returncode}")
print("CBA step finished.")

[tornado] Filtering ATTRIBUTE_STRATEGY from 64 to 2 rows for strategy_code(s): ['AGRC:DEC_CH4_RICE', 'BASE'] (including BASE)
Cargamos configuración de archivo excel
Se actualizó la base de datos

************************************
*Strategy : AGRC:DEC_CH4_RICE (0/1)
************************************

---------Costs for: cb:wali:technical_cost:sanitation:unimp_rural.
La variable se evalúa en System Cost
---------Costs for: cb:wali:technical_cost:sanitation:imp_rural.
La variable se evalúa en System Cost
---------Costs for: cb:wali:technical_cost:sanitation:safeman_rural.
La variable se evalúa en System Cost
---------Costs for: cb:wali:technical_cost:sanitation:unimp_urban.
La variable se evalúa en System Cost
---------Costs for: cb:wali:technical_cost:sanitation:imp_urban.
La variable se evalúa en System Cost
---------Costs for: cb:wali:technical_cost:sanitation:safeman_urban.
La variable se evalúa en System Cost
---------Costs for: cb:wali:technical_cost:sanitation:omit_rural.
La

## 3. Run tornado and whirlpool processes

Executes `scr/process_tornado.py` and `scr/process_whirlpool.py` with the parameters set above.

In [14]:
for label, script, input_dir, output_dir in [
    ("tornado", PROCESS_TORNADO_SCRIPT, INPUT_TORNADO, PATH_OUTPUT / "tornado"),
    ("whirlpool", PROCESS_WHIRLPOOL_SCRIPT, INPUT_WHIRLPOOL, PATH_OUTPUT / "whirlpool"),
]:
    if not script.exists():
        raise FileNotFoundError(f"Script not found: {script}")
    out = subprocess.run(
        [sys.executable, str(script), "--country", COUNTRY, "--input-dir", str(input_dir), "--output-dir", str(output_dir), "--emissions-year", str(EMISSIONS_YEAR)],
        cwd=str(PROJECT_ROOT),
        capture_output=True,
        text=True,
    )
    if out.returncode != 0:
        print(out.stdout)
        print(out.stderr)
        raise RuntimeError(f"{label} script failed with code {out.returncode}")
    print(f"{label} step finished.")
print("Both process scripts finished.")

tornado step finished.
whirlpool step finished.
Both process scripts finished.


## 4. Evaluation: marginal_total_abatement_cost_(USD/tCO2e)

Load the saved tables, extract the marginal abatement cost column and key identifiers, and write a **evaluation CSV** plus a **summary** for quick comparison.

In [15]:
def load_and_tag(path: pathlib.Path, process_label: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    if MAC_COL not in df.columns:
        raise KeyError(f"Column '{MAC_COL}' not found in {path}. Columns: {list(df.columns)}")
    df["process"] = process_label
    return df

EVALUATION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

dfs = []
if OUTPUT_TORNADO_CSV.exists():
    dt = load_and_tag(OUTPUT_TORNADO_CSV, "tornado")
    dfs.append(dt)
if OUTPUT_WHIRLPOOL_CSV.exists():
    dw = load_and_tag(OUTPUT_WHIRLPOOL_CSV, "whirlpool")
    dfs.append(dw)

if not dfs:
    raise FileNotFoundError(
        f"Neither {OUTPUT_TORNADO_CSV} nor {OUTPUT_WHIRLPOOL_CSV} found. Run the tornado and whirlpool notebooks first."
    )

combined = pd.concat(dfs, ignore_index=True)

In [16]:
# Key columns to keep for evaluation (adjust if your tables differ)
id_cols = [c for c in ["primary_id", "strategy", "sector", "transformation_name", "process"] if c in combined.columns]
eval_cols = id_cols + [MAC_COL]
eval_df = combined[[c for c in eval_cols if c in combined.columns]].copy()
eval_df = eval_df.drop_duplicates()
eval_df.to_csv(EVALUATION_CSV, index=False)
print(f"Evaluation table saved to: {EVALUATION_CSV}")
eval_df.head(10)

Evaluation table saved to: /Users/alexa/Projects/ssp_uganda_data/ssp_modeling/cb/tornado_plot/data/output/marginal_abatement_cost_evaluation.csv


,primary_id,strategy,sector,transformation_name,process,marginal_total_abatement_cost_(USD/tCO2e)
0,1001.0,Singleton - Default Value - AGRC: Improve rice...,AGRC,Improve rice management,tornado,2.219182e+10
1,76076.0,Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from NZ,AGRC,DEC_CH4_RICE_STRATEGY_NZ,whirlpool,1.131946e+12


In [17]:
# Summary stats by process (and by strategy/sector if present)
group_cols = [c for c in ["process", "strategy", "sector"] if c in eval_df.columns]
if not group_cols:
    summary = eval_df[MAC_COL].agg(["count", "min", "max", "mean", "median", "std"]).to_frame().T
else:
    summary = (
        eval_df.groupby(group_cols, dropna=False)[MAC_COL]
        .agg(["count", "min", "max", "mean", "median"])
        .reset_index()
    )
summary.to_csv(EVALUATION_SUMMARY_CSV, index=False)
print(f"Summary saved to: {EVALUATION_SUMMARY_CSV}")
summary

Summary saved to: /Users/alexa/Projects/ssp_uganda_data/ssp_modeling/cb/tornado_plot/data/output/marginal_abatement_cost_summary.csv


,process,strategy,sector,count,min,max,mean,median
0,tornado,Singleton - Default Value - AGRC: Improve rice...,AGRC,1,2.219182e+10,2.219182e+10,2.219182e+10,2.219182e+10
1,whirlpool,Remove TX:AGRC:DEC_CH4_RICE_STRATEGY_NZ from NZ,AGRC,1,1.131946e+12,1.131946e+12,1.131946e+12,1.131946e+12
